In [ ]:
import aiohttp
import asyncio
import nltk
from nltk.tokenize import sent_tokenize
import redis
import logging
from datetime import datetime

# 初始化配置（证据1、2、10、14）
WIKI_API = "https://en.wikipedia.org/w/api.php"
CONCURRENCY_LIMIT = 500  # 并发限制500/秒（证据1）
REDIS_HOST = 'localhost'
REDIS_PORT = 6379
nltk.download('punkt')  # 下载分句模型（证据2）

class WikiProcessor:
    def __init__(self, keyword, max_sentences=1_000_000):
        self.keyword = keyword
        self.max_sentences = max_sentences
        self.redis_pool = redis.ConnectionPool(host=REDIS_HOST, port=REDIS_PORT)
        self.semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)  # 控制并发（证据1）
        self.counter = 0
    
    async def fetch_articles(self, session):
        """异步获取Wiki文章列表（证据1）"""
        params = {
            "action": "query",
            "list": "search",
            "srsearch": self.keyword,
            "format": "json",
            "srlimit": 500  # 每页最大结果数
        }
        async with session.get(WIKI_API, params=params) as response:
            data = await response.json()
            return [item['title'] for item in data['query']['search']]

    async def process_article(self, session, title):
        """处理单篇文章（证据1、6、9）"""
        async with self.semaphore:  # 并发控制
            params = {
                "action": "query",
                "prop": "extracts",
                "titles": title,
                "explaintext": 1,
                "format": "json"
            }
            try:
                async with session.get(WIKI_API, params=params) as response:
                    data = await response.json()
                    page = next(iter(data['query']['pages'].values()))
                    return page.get('extract', '')
            except Exception as e:
                logging.error(f"Error processing {title}: {e}")
                return ''

    def is_valid_sentence(self, sentence):
        """过滤无效句子（证据2、19）"""
        return len(sentence) > 20 and self.keyword.lower() in sentence.lower()

    async def save_to_redis(self, sentences):
        """批量写入Redis（证据10、14）"""
        r = redis.Redis(connection_pool=self.redis_pool)
        with r.pipeline() as pipe:
            for sent in sentences:
                pipe.sadd(self.keyword, sent)  # 使用集合去重
            pipe.execute()

    async def run(self):
        start = datetime.now()
        async with aiohttp.ClientSession() as session:
            # 获取相关文章列表
            articles = await self.fetch_articles(session)
            
            # 异步处理所有文章
            tasks = [self.process_article(session, title) for title in articles]
            for future in asyncio.as_completed(tasks):
                content = await future
                if not content: continue
                
                # 分句和过滤
                sentences = [s for s in sent_tokenize(content) if self.is_valid_sentence(s)]
                if not sentences: continue
                
                # 存储到Redis并计数
                await self.save_to_redis(sentences)
                self.counter += len(sentences)
                print(f"Processed {self.counter}/{self.max_sentences} sentences")
                
                # 达到数量限制时终止
                if self.counter >= self.max_sentences:
                    break
        
        print(f"Finished in {(datetime.now()-start).total_seconds():.2f}s")

if __name__ == "__main__":
    processor = WikiProcessor(keyword="Python", max_sentences=1000)
    asyncio.run(processor.run())


tsp python knowledge/wiki_test.py --keyword Medicine
tsp python knowledge/wiki_test.py --keyword Finance
tsp python knowledge/wiki_test.py --keyword Law
tsp python knowledge/wiki_test.py --keyword News
